# Capa socioeconómica: correspondencia circuito↔radio censal y EPH Gran La Plata

Arma los dos insumos de la capa socioeconómica que sí se pueden automatizar (ver el plan de trabajo):

1. **`circuito_radio_correspondencia.csv`**: reparte cada radio censal (2010 y 2022) entre los circuitos electorales de La Plata que intersecta, ponderado por área — prerrequisito técnico para unir cualquier variable de Censo a la tabla electoral (`src/socioeconomia/geo.py`).
2. **`eph_gran_la_plata.csv`**: serie trimestral 2017T2-2025T4 de desocupación/informalidad/ingreso para el aglomerado Gran La Plata (`src/socioeconomia/eph_client.py`) — a nivel de aglomerado, **nunca** de circuito (ver limitación al final).

Lo que **no** arma este notebook: `censo_2010_radio.csv`/`censo_2022_radio.csv` (país de nacimiento, nivel educativo, condición de actividad, vivienda por radio) — esa extracción es manual en REDATAM

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))

from socioeconomia.eph_client import EphClient, agregados_gran_la_plata, TrimestreNoPublicado, UrlDesconocida
from socioeconomia.geo import calcular_correspondencia, cargar_circuitos_electorales, cargar_radios_censales

SOCIOECONOMIA = REPO / "data" / "socioeconomia"

## 1. Correspondencia espacial circuito electoral ↔ radio censal

Circuitos electorales (`circuitos_electorales_la_plata.geojson`, Cámara Nacional Electoral / catálogo PBA) y radios censales (`radios_censales_2010_la_plata.geojson` / `_2022_la_plata.geojson`, cartografía armonizada CONICET 1991/2001/2010/2022) no comparten identificador — el join es espacial, no por id (ver `src/socioeconomia/geo.py`). Cada radio prorrateado entre varios circuitos queda con varias filas cuyo `peso_area` suma 1.0; `match_limpio=True` si el radio cayó entero dentro de un único circuito.

In [2]:
circuitos = cargar_circuitos_electorales(SOCIOECONOMIA / "circuitos_electorales_la_plata.geojson")
print(f"circuitos electorales de La Plata: {len(circuitos)}")

partes = []
for anio, nombre_archivo in [(2010, "radios_censales_2010_la_plata.geojson"), (2022, "radios_censales_2022_la_plata.geojson")]:
    radios = cargar_radios_censales(SOCIOECONOMIA / nombre_archivo, anio)
    print(f"radios censales {anio}: {len(radios)}")
    partes.append(calcular_correspondencia(circuitos, radios))

correspondencia = pd.concat(partes, ignore_index=True)
destino = SOCIOECONOMIA / "circuito_radio_correspondencia.csv"
correspondencia.to_csv(destino, index=False)
print(f"\n{len(correspondencia)} filas -> {destino}")

circuitos electorales de La Plata: 68
radios censales 2010: 849


radios censales 2022: 1049



3057 filas -> /workspaces/analisis-politica-economia/data/socioeconomia/circuito_radio_correspondencia.csv


In [3]:
# Cobertura: cuántos radios de cada censo tienen match limpio (un solo circuito) vs. prorrateado.
# Un radio prorrateado no es un error -- es real que los límites de circuito y de radio censal no
# coinciden -- pero cualquier cifra por circuito construida sobre esas filas es una estimación por
# área, no un conteo censal (ver README).
for anio in sorted(correspondencia["censo_anio"].unique()):
    sub = correspondencia[correspondencia["censo_anio"] == anio]
    n_radios = sub["radio_censal_id"].nunique()
    n_limpios = sub.loc[sub["match_limpio"], "radio_censal_id"].nunique()
    n_circuitos = sub["circuito_id"].nunique()
    print(f"{anio}: {n_radios} radios, {n_limpios} ({n_limpios/n_radios:.1%}) con match limpio, "
          f"{n_circuitos}/{len(circuitos)} circuitos con al menos un radio asociado")

circuitos_limite_incierto = {"493", "496F", "504C"}  # ya señalados en README (§ circuito_id canónico)
presentes = set(circuitos["circuito_id"])
print(f"\ncircuitos de límite incierto presentes en la capa de circuitos electorales: "
      f"{sorted(circuitos_limite_incierto & presentes)}")
print(f"ausentes de la capa (no se pudo bajar su polígono): {sorted(circuitos_limite_incierto - presentes)}")

2010: 849 radios, 454 (53.5%) con match limpio, 68/68 circuitos con al menos un radio asociado
2022: 1049 radios, 585 (55.8%) con match limpio, 68/68 circuitos con al menos un radio asociado

circuitos de límite incierto presentes en la capa de circuitos electorales: ['493', '496F']
ausentes de la capa (no se pudo bajar su polígono): ['504C']


## 2. EPH Gran La Plata — serie trimestral

`AGLOMERADO=2` (confirmado empíricamente: en la base individual del 1er trimestre de 2018 concentra 870.693 personas ponderadas, en línea con la población conocida de Gran La Plata). Cubre 2017T2-2025T4 con el patrón de URL regular de INDEC — antes de 2017T2 el nombre de archivo no sigue un patrón fijo y hay que pasar `url` a mano (ver docstring de `eph_client.py`); esos trimestres quedan afuera de este notebook por ahora.

In [4]:
client = EphClient(cache_dir=SOCIOECONOMIA / "eph_cache")

filas = []
for anio in range(2017, 2026):
    for trimestre in (1, 2, 3, 4):
        if anio == 2017 and trimestre == 1:
            continue  # sin URL confirmada, ver docstring de eph_client.py
        try:
            zip_path = client.descargar_trimestre(anio, trimestre)
            individual = client.leer_base(zip_path, "individual")
            hogar = client.leer_base(zip_path, "hogar")
            filas.append(agregados_gran_la_plata(individual, hogar))
        except (TrimestreNoPublicado, UrlDesconocida) as e:
            print(f"{anio} T{trimestre}: {e}")

eph_gran_la_plata = pd.DataFrame(filas).sort_values(["anio", "trimestre"]).reset_index(drop=True)
destino_eph = SOCIOECONOMIA / "eph_gran_la_plata.csv"
eph_gran_la_plata.to_csv(destino_eph, index=False)
print(f"\n{len(eph_gran_la_plata)} trimestres -> {destino_eph}")
eph_gran_la_plata.tail()


35 trimestres -> /workspaces/analisis-politica-economia/data/socioeconomia/eph_gran_la_plata.csv


,anio,trimestre,tasa_desocupacion,tasa_informalidad,ingreso_ocupacion_principal_medio,ipcf_medio
30,2024,4,0.080693,0.286565,540574.437126,602173.349588
31,2025,1,0.086728,0.338175,571092.938179,704607.202897
32,2025,2,0.069100,0.280446,650116.588518,643531.542057
33,2025,3,0.081167,0.269054,652146.499571,757610.648866
34,2025,4,0.094811,0.230054,638453.200880,725480.853385


## 3. Join con Censo por circuito (pendiente de la extracción manual REDATAM)

`censo_2010_radio.csv` / `censo_2022_radio.csv` no existen todavía en este repositorio — su extracción es manual (ver `data/socioeconomia/EXTRACCION_REDATAM.md`). Esta celda deja armado el join que hay que correr apenas existan: cada variable censal por radio se multiplica por `peso_area` antes de sumar por circuito, para que los radios prorrateados aporten solo la porción de su área que cae en ese circuito.

In [5]:
def unir_censo_a_circuitos(censo_radio: pd.DataFrame, censo_anio: int, columnas_variables: list[str]) -> pd.DataFrame:
    """censo_radio: una fila por radio_censal_id, con las columnas de columnas_variables ya numéricas."""
    corr = correspondencia[correspondencia["censo_anio"] == censo_anio]
    unido = corr.merge(censo_radio, on="radio_censal_id", how="left")
    for col in columnas_variables:
        unido[col] = unido[col] * unido["peso_area"]
    return unido.groupby("circuito_id")[columnas_variables].sum().reset_index()

for anio, nombre in [(2010, "censo_2010_radio.csv"), (2022, "censo_2022_radio.csv")]:
    ruta = SOCIOECONOMIA / nombre
    if ruta.exists():
        censo_radio = pd.read_csv(ruta, dtype={"radio_censal_id": str})
        columnas_variables = [c for c in censo_radio.columns if c != "radio_censal_id"]
        censo_por_circuito = unir_censo_a_circuitos(censo_radio, anio, columnas_variables)
        destino = SOCIOECONOMIA / f"censo_{anio}_circuito.csv"
        censo_por_circuito.to_csv(destino, index=False)
        print(f"censo {anio}: {len(censo_por_circuito)} circuitos -> {destino}")
    else:
        print(f"censo {anio}: falta {ruta.name} (extracción manual pendiente, ver EXTRACCION_REDATAM.md)")

censo 2010: falta censo_2010_radio.csv (extracción manual pendiente, ver EXTRACCION_REDATAM.md)
censo 2022: falta censo_2022_radio.csv (extracción manual pendiente, ver EXTRACCION_REDATAM.md)


## Limitaciones 

1. **EPH = aglomerado Gran La Plata (La Plata+Berisso+Ensenada), nunca por circuito.** Cualquier análisis que cruce la serie EPH con resultados electorales por circuito está mezclando grados de agregación distintos — señalarlo explícitamente, no forzarlo a un único denominador.
2. **Censo 2022 describe la estructura cerca de 2022**, no las elecciones tempranas de este proyecto (2011-2015) — no usarlo para explicarlas sin decirlo.
3. **Los radios prorrateados no son un caso raro**: bastante más de un tercio de los radios de La Plata (ver cobertura arriba) cruzan el límite de más de un circuito. Cualquier cifra censal por circuito construida a partir de esas filas es una estimación por área, no un conteo censal.
4. **Comparar Censo 2010 vs. 2022 exige pasar por la geometría, no por el `radio_censal_id` crudo** — los radios cambian de límites entre censos (ver `EXTRACCION_REDATAM.md`).